In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import rasterio as rio

In [ ]:
from scipy.stats import pearsonr

In [ ]:
import geopandas as gpd

In [ ]:
import pandas as pd

In [ ]:
from shapely import Point
import numpy as np

In [ ]:
import os

In [ ]:
out_path=r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\kamareddy'

In [ ]:
def get_coord(shape):
    shapefile=pd.read_excel(shape)
    coords = [(x,y) for x, y in zip(shapefile.Longitude, shapefile.Latitude)]
    return coords

In [ ]:
def getRasterValue(image,coords):
    ras = rio.open(image)
    return [x[0] for x in ras.sample(coords)]

In [ ]:
def mape(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


In [ ]:
def accuracy_parameter(df):
    y_test=np.array(df['Yield (kg_p_ha)'])
    y_pred=np.array(df['Predicted'])
    mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
    mse = mean_squared_error(y_true=y_test,y_pred=y_pred) #default=True
    rmse = mean_squared_error(y_true=y_test,y_pred=y_pred,squared=False)
    if len(y_test)>2:
        r2 = pearsonr(y_test,y_pred)[0]
    else:
        r2=0
    maper = mape(y_test,y_pred)
    test_mean=y_test.mean()
    pred_mean=y_pred.mean()
    # print("MAE:",round(mae,2))
    # print("MSE:",round(mse,2))
    # print("RMSE:",round(rmse,2))
    # print("R2:",round(r2,2))
    # print("MAPE:",round(maper,2))
    
    return mae,mse,rmse,r2,maper,test_mean, pred_mean

In [ ]:
raster = r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\kamareddy\output\Kamareddy_maize_yield.tif'
shapefile = r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\zips\GP shape\Kamareddy.shp'
points = (os.path.join(out_path,'cce','Kamareddy.xlsx'))

In [ ]:
shp = gpd.read_file(shapefile)
# pts = pd.read_excel(points)
pts = pd.read_excel((os.path.join(out_path,'predic_actual_gp.xlsx')))

In [ ]:
geometry = [Point(xy) for xy in zip(pts['Longitude'], pts['Latitude'])]
pt_gdf = gpd.GeoDataFrame(pts, crs='EPSG:4326', geometry=geometry)
shp = shp.to_crs(pt_gdf.crs)
# pt_gdf['Predicted']=getRasterValue(raster,get_coord(points))
# pt_gdf=pt_gdf[pt_gdf['Predicted']>0]
# pt_gdf['difference']=abs(pt_gdf['Predicted']-pt_gdf['Yield(kg_p_ha)'])
# pt_gdf=pt_gdf[pt_gdf['difference']<500]

In [ ]:
import seaborn as sns

In [ ]:
test_df = pts

In [ ]:
test_df = gpd.sjoin(pt_gdf,shp,how='inner', predicate='intersects')

In [ ]:
##### yield kg_p_ha

test_df['Yield (kg_p_ha)']=test_df['Yield__kg_']*0.86
test_df['Predicted'] = test_df['Predicted']*0.86

In [ ]:
test_df=test_df[test_df['Predicted']>0]

In [ ]:
shp.groupby('GP_Name').count().shape[0]

In [ ]:
test_df

In [ ]:
grp = test_df.groupby('GP_Name')
data=[]
npts=[]
fid=[]
for d in grp:
    npts.append(d[1].shape[0])
    data.append(accuracy_parameter(d[1]))
    fid.append(d[0])

In [ ]:
df=pd.DataFrame(data,columns=['MAE','MSE','RMSE','R','MAPE','Observed','Predicted'])

In [ ]:
df['GP_Name']=fid

In [ ]:
# shp_map=shp[['GPNAME','OBJECTID']]
# df_gp=pd.merge(df,shp_map,on='OBJECTID')
df_gp=df

In [ ]:
df_gp

In [ ]:
# df_gp[df_gp['R']>=0]

In [ ]:
df_gp1=df_gp[df_gp['MAPE']<50]

In [ ]:
df_gp1=df_gp1.drop(columns=['R','MSE'])

In [ ]:
df_gp1.to_excel(os.path.join(out_path,'kamareddy_GP_mc.xlsx'))